# SatQuery AI — MAIN server (Kaggle T4)

Runs the same FastAPI backend as the laptop, with Ollama `qwen3.5:2b` on the T4, behind a free cloudflared quick tunnel.

**Notebook settings (right panel):** Accelerator = **GPU T4**, Internet = **On** (needs a phone-verified Kaggle account).

**Get the code here, one of:**
- **GitHub (public repo):** set `REPO_URL` in the next cell. Note `web/.gitignore` drops `web/dist`, so the tunnel then serves only the API (the laptop UI is what you use anyway).
- **Kaggle Dataset:** on the laptop, in the project folder run
  `tar -a -cf satquery-app.zip --exclude=__pycache__ satquery web/dist requirements.txt`
  then kaggle.com → Datasets → New Dataset → upload the zip. In this notebook: **Add Input** → that dataset. Leave `REPO_URL` empty.

**Then:** Run All. The tunnel cell prints `MAIN URL: https://….trycloudflare.com` and a `TOKEN`. On the laptop UI click **Server** (top right), paste both, save. The status should read **MAIN qwen3.5:2b online**. The URL changes every run.

Keep the last cell running: it is the keep-alive. Interactive sessions can idle-stop; for an unattended demo use **Save Version → Save & Run All** (up to 12 h on GPU, URL appears in the version log). GPU quota is ~30 h/week per account.
The tunnel URL is public, so `/query` needs the printed token (a new one every run); `/health` stays open. Share both only with the team.

In [ ]:
REPO_URL = ""             # e.g. "https://github.com/<you>/<repo>.git" (public); empty = use the attached Kaggle Dataset
VLM_MODEL = "qwen3.5:2b"  # same as LITE; the T4 has 16 GB, so a bigger Ollama tag also fits (swap to the M1 fine-tune later)

In [ ]:
import glob, os, pathlib, re, subprocess, sys, time
W = pathlib.Path("/kaggle/working")
sh = lambda cmd: subprocess.run(cmd, shell=True, check=True)
log = lambda name: open(W / f"{name}.log", "w")

if REPO_URL:
    APP = W / "app"
    if not APP.exists():
        sh(f"git clone --depth 1 {REPO_URL} {APP}")
else:
    hits = glob.glob("/kaggle/input/**/satquery/server.py", recursive=True)
    assert len(hits) == 1, f"need exactly one attached satquery-app dataset, found {hits}"
    APP = pathlib.Path(hits[0]).parent.parent  # read-only is fine, uvicorn only reads
print("code:", APP, "| UI bundled:", (APP / "web/dist/index.html").exists())
sh(f"grep -v gradio {APP}/requirements.txt | pip install -q -r /dev/stdin")  # ponytail: gradio is the laptop fallback UI only
import httpx


def wait(url, secs=120):
    for _ in range(secs):
        try:
            if httpx.get(url, timeout=5).is_success:
                return
        except httpx.HTTPError:
            pass
        time.sleep(1)
    raise RuntimeError(f"{url} not up after {secs}s; read the *.log files in /kaggle/working")

In [ ]:
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
if not os.path.exists("/usr/local/bin/ollama"):
    sh("command -v zstd || (apt-get -qq update && apt-get -qq install -y zstd)")  # Ollama ships .tar.zst
    sh("curl -fsSL https://ollama.com/install.sh | sh")  # warns 'systemd not running': fine, we start it below
ollama = subprocess.Popen(["ollama", "serve"], env={**os.environ, "OLLAMA_KEEP_ALIVE": "-1"},
                          stdout=log("ollama"), stderr=subprocess.STDOUT)
wait("http://localhost:11434/api/version")
sh(f"ollama pull {VLM_MODEL}")
httpx.post("http://localhost:11434/api/generate", json={"model": VLM_MODEL, "keep_alive": -1}, timeout=600)  # load into VRAM now
sh("ollama ps")  # PROCESSOR column should say 100% GPU

In [ ]:
import secrets
# The tunnel is public: /query needs this token. A Kaggle secret named SATQUERY_TOKEN (Add-ons -> Secrets) keeps it
# out of the notebook output; without one a fresh token is made and printed (interactive sessions only).
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN, TOKEN_SRC = UserSecretsClient().get_secret("SATQUERY_TOKEN"), "Kaggle secret SATQUERY_TOKEN"
except Exception:
    TOKEN, TOKEN_SRC = secrets.token_urlsafe(18), None
    assert os.environ.get("KAGGLE_KERNEL_RUN_TYPE") != "Batch", "Save & Run All would store the token in the version log: add a SATQUERY_TOKEN Kaggle secret first"
api = subprocess.Popen([sys.executable, "-m", "uvicorn", "satquery.server:app", "--host", "127.0.0.1", "--port", "8000"],
                       cwd=APP, stdout=log("uvicorn"), stderr=subprocess.STDOUT,
                       env={**os.environ, "VLM_URL": "http://localhost:11434/v1", "VLM_MODEL": VLM_MODEL,
                            "VLM_REASONING": "none", "SATQUERY_TOKEN": TOKEN})  # none: qwen3.5 otherwise spends all tokens thinking
wait("http://localhost:8000/health")
print(httpx.get("http://localhost:8000/health").json())
# fail closed: a stale dataset (pre-token server.py) would ignore SATQUERY_TOKEN and go public without auth
r = httpx.post("http://localhost:8000/query", data={"q": "x"}, files={"files": ("a.png", b"x")})
assert r.status_code == 401, f"loaded code does not enforce SATQUERY_TOKEN (got {r.status_code}): upload a new dataset version"

In [ ]:
CF = W / "cloudflared"
if not CF.exists():
    sh(f"curl -fsSL -o {CF} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x {CF}")
tunnel = subprocess.Popen([str(CF), "tunnel", "--no-autoupdate", "--url", "http://localhost:8000"],
                          stdout=log("cloudflared"), stderr=subprocess.STDOUT)


def tunnel_url(text):
    m = re.search(r"https://(?!api\.)[-a-z0-9]+\.trycloudflare\.com", text)  # api.trycloudflare.com shows up in errors
    return m and m.group(0)


for _ in range(60):
    URL = tunnel_url((W / "cloudflared.log").read_text())
    if URL:
        break
    time.sleep(1)
assert URL, "no tunnel URL; read /kaggle/working/cloudflared.log"
wait(URL + "/health", 180)  # new trycloudflare hostnames can take a minute to resolve
h = httpx.get(URL + "/health", timeout=10).json()
print(h)
print(f"\nMAIN URL: {URL}\nTOKEN:    {TOKEN_SRC or TOKEN}\nPaste both into the laptop UI (Server button). Expect vlm_online=True: {h['vlm_online']}")

In [ ]:
# Keep-alive + watchdog. Leave running; interrupt (or stop the session) to shut MAIN down.
while True:
    dead = [n for n, p in (("ollama", ollama), ("uvicorn", api), ("cloudflared", tunnel)) if p.poll() is not None]
    try:
        h = httpx.get(URL + "/health", timeout=15).json()
        print(time.strftime("%H:%M"), URL, "vlm_online:", h["vlm_online"], "dead:", dead or "-", flush=True)
    except (httpx.HTTPError, ValueError) as e:
        print(time.strftime("%H:%M"), "health failed:", e, "dead:", dead or "-", flush=True)
    time.sleep(300)